# Tarea 9.10



Sea $P(t)$ la cantidad de individuos en el tiempo $t$, medido en años. La razón de crecimiento está dada por

$$\frac{dP}{dt} = bP(t) - k[P(t)]^2$$

Cada término tiene un significado claro:

- $bP(t)$ son los **nacimientos**. Mientras más individuos hay, más nacimientos ocurren.
- $kP(t)^2$ son las **muertes por hacinamiento**. El enunciado dice que la tasa de muerte es $d = kP(t)$, así que las muertes totales son $dP = kP^2$. Este término crece más rápido que $P$ cuando la población es grande.

Los datos son:

- $P(0) = 50976$
- $b = 2.9 \times 10^{-2}$
- $k = 1.4 \times 10^{-7}$
- Queremos $P(5)$, la población a los 5 años.

### Una estimación antes de calcular

Si la población se estabiliza, $dP/dt = 0$, es decir $bP = kP^2$, y entonces

$$P^* = \frac{b}{k} = \frac{2.9 \times 10^{-2}}{1.4 \times 10^{-7}} \approx 207\,142.86$$

Como $P(0) = 50\,976$ está por debajo de $P^*$, la población debería crecer hacia ese valor. Así que a los 5 años esperamos algo un poco mayor que 50 976, y muy por debajo de 207 000. Si el resultado no cumple esto, hay un error.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Datos del problema 
P0 = 50976        # población inicial P(0)
b  = 2.9e-2       # tasa de natalidad
k  = 1.4e-7       # constante de hacinamiento
a  = 0.0          # tiempo inicial (años)
B  = 5.0          # tiempo final (años)

print(f"Punto de equilibrio P* = b/k = {b / k:,.2f}")

In [ ]:
def f(t, P):
    # Lado derecho de la EDO: dP/dt = f(t, P)
    return b * P - k * P**2

print(f"f(0, P*) = {f(0, b / k):.2e}  (debe ser ~0, porque en el equilibrio no hay cambio)")
print(f"f(0, P0) = {f(0, P0):.2f}  (positivo, así que la población crece)")


Un método de un paso, como Euler o Runge-Kutta, calcula el siguiente valor usando solo el último punto. Pero un metodo multi pasos reutiliza los últimos puntos, que ya tienen calculada su pendiente, y así aprovecha mejor la información.

El método de **Adams** tiene dos partes:

**Predictor (Adams-Bashforth de 4 pasos)**: extrapola una primera aproximación $P^*_{i+1}$:

$$P^*_{i+1} = P_i + \frac{h}{24}\left[55 f_i - 59 f_{i-1} + 37 f_{i-2} - 9 f_{i-3}\right]$$

**Corrector (Adams-Moulton de 4 puntos)**: usa $P^*_{i+1}$ dentro de la fórmula para corregir el valor:

$$P_{i+1} = P_i + \frac{h}{24}\left[9 f(t_{i+1}, P^*_{i+1}) + 19 f_i - 5 f_{i-1} + f_{i-2}\right]$$

Aquí $f_i = f(t_i, P_i)$ y $h$ es el tamaño de paso.

Como el método necesita 4 puntos para comenzar, los **primeros 3 pasos** se calculan con **Runge-Kutta de orden 4**.



Este es un paso del método de Runge-Kutta de orden 4. Devuelve el incremento que hay que sumarle a $P$.

In [ ]:
def rk4_step(f, t, w, h):
    # Incremento de un paso de Runge-Kutta de orden 4
    k1 = h * f(t, w)
    k2 = h * f(t + h/2, w + k1/2)
    k3 = h * f(t + h/2, w + k2/2)
    k4 = h * f(t + h,   w + k3)
    return (k1 + 2*k2 + 2*k3 + k4) / 6



En cada iteración el código:

1. Asigna el tiempo del nuevo punto, $t_{i+1}$.
2. Predice $P^*_{i+1}$ con Adams-Bashforth.
3. Corrig con Adams-Moulton, evaluando $f$ en la predicción.
4. Guarda el nuevo valor y calcula su pendiente para los pasos siguientes.

El orden importa: el tiempo $t_{i+1}$ debe existir antes de usarlo en el corrector. 

In [ ]:
def adams_pc4(f, t0, T, y0, N):
    # Resuelve y' = f(t, y) en [t0, T] con N pasos 
    h = (T - t0) / N

    t = np.zeros(N + 1)
    w = np.zeros(N + 1)
    t[0] = t0
    w[0] = y0

    #primeros 3 pasos con RK4
    for i in range(3):
        t[i+1] = t[i] + h
        w[i+1] = w[i] + rk4_step(f, t[i], w[i], h)

    # Pendientes de los primeros 4 puntos
    F = [f(t[i], w[i]) for i in range(4)]

    #Pasos de Adams
    for i in range(3, N):
        t[i+1] = t[i] + h

        # Predictor (Adams-Bash))
        wp = w[i] + h * (55*F[i] - 59*F[i-1] + 37*F[i-2] - 9*F[i-3]) / 24

        # Corrector (Adams-Moulton)
        wc = w[i] + h * (9*f(t[i+1], wp) + 19*F[i] - 5*F[i-1] + F[i-2]) / 24

        w[i+1] = wc
        F.append(f(t[i+1], w[i+1]))

    return t, w



La ecuación logística tiene solución cerrada, lo que nos permite verificar nuestro método:

$$P(t) = \frac{b\,P_0}{k\,P_0 + (b - k\,P_0)\,e^{-bt}}$$

In [ ]:
def exacta(t):
    # Solución analítica de la ecuación logística
    return (b * P0) / (k * P0 + (b - k * P0) * np.exp(-b * t))



Finalmente usamos $N = 50$ pasos, es decir $h = 0.1$ años.

In [ ]:
N = 50
h = (B - a) / N

ts, Ps = adams_pc4(f, a, B, P0, N)
error = np.abs(Ps - exacta(ts))

print(f"Tamaño de paso h = {h}")
print(f"P(5) numérica = {Ps[-1]:,.4f}")
print(f"P(5) exacta   = {exacta(B):,.4f}")
print(f"Error absoluto = {error[-1]:.2e}")

### Entonces, ¿Qué tan rápido mejora el método?

Un método de orden 4 debería reducir el error de forma muy notable al dividir el paso. Para medirlo, usamos el **orden empírico**:

$$p \approx \frac{\ln(e_{\text{anterior}}/e_{\text{actual}})}{\ln(N_{\text{actual}}/N_{\text{anterior}})}$$

Para $N$ grandes el error llega a unos $10^{-11}$, que es del tamaño del redondeo de la computadora, así que el orden en esos casos ya no es confiable. Por eso lo estimamos con los primeros valores.

In [ ]:
Ns = [10, 20, 50, 100, 200]
print(f"{'N':>6} {'h':>10} {'P(5) numérica':>18} {'Error absoluto':>16} {'Orden':>8}")

errores = []
Nprev = None
for N_test in Ns:
    _, P_test = adams_pc4(f, a, B, P0, N_test)
    err = abs(P_test[-1] - exacta(B))
    orden = ""
    if errores:
        orden = f"{np.log(errores[-1] / err) / np.log(N_test / Nprev):.2f}"
    errores.append(err)
    Nprev = N_test
    print(f"{N_test:>6} {(B-a)/N_test:>10.4f} {P_test[-1]:>18,.4f} {err:>16.2e} {orden:>8}")

### Gráficamente

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))

tt = np.linspace(a, B, 400)
ax[0].plot(tt, exacta(tt), 'k-', label='Solución exacta')
ax[0].plot(ts, Ps, 'o--', markersize=4, label=f'Adams PC4 (N = {N})')
ax[0].set_xlabel('t (años)')
ax[0].set_ylabel('P(t)')
ax[0].set_title('Crecimiento de la población')
ax[0].legend()
ax[0].grid(alpha=0.3)

ax[1].semilogy(ts, error + 1e-16, 'o-', markersize=4, color='tab:red')
ax[1].set_xlabel('t (años)')
ax[1].set_ylabel('Error absoluto (escala log)')
ax[1].set_title('Error a lo largo del tiempo')
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()



Con los datos del problema, la población a los 5 años es el valor que muestra la celda siguiente. Está entre $P(0) = 50\,976$ y el equilibrio $P^* \approx 207\,143$, como anticipamos al principio, y coincide con la solución exacta con un error muy pequeño.

In [ ]:
print(f"La población a los 5 años es P(5) ≈ {Ps[-1]:,.0f} individuos (método de Adams PC4).")
print(f"Valor exacto de referencia: {exacta(B):,.0f} individuos.")
print(f"Error absoluto: {error[-1]:.2e}")

### Conclusión

Con el método de Adams predictor-corrector de 4 pasos (arrancado con RK4) obtuvimos P(5) ≈ 56 751 individuos, con un error absoluto de 3.46·10⁻⁹ respecto a la solución exacta. El orden se mantiene cerca de 4 para N = 20, 50 y 100. Para N = 200 el error ya está en el límite del redondeo.

El resultado parece ser realista, la población crece desde 50 976 hacia la capacidad de carga P* ≈ 207 143, pero en 5 años solo recorre una parte de ese camino.